In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import keras
import torch
import pandas as pd
import numpy as np
from keras import layers
from pathlib import Path
keras.utils.set_random_seed(58008)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
torch.cuda.empty_cache()

In [ ]:
train_paths = list(Path('./data/train').glob('*.jpg'))
train_df  = pd.DataFrame({
    'filepath': [str(p) for p in train_paths],
    'label': [1 if p.stem.startswith('dog') else 0 for p in train_paths]
})
train_df = train_df.sample(frac=1, random_state = 58008).reset_index(drop=True)

In [ ]:
from PIL import Image
sizes = [Image.open(p).size for p in train_paths[:500]]
widths, heights = zip(*sizes)
print(f"width:  min={min(widths)} max={max(widths)} avg={sum(widths)//len(widths)}")
print(f"height: min={min(heights)} max={max(heights)} avg={sum(heights)//len(heights)}")

width:  min=66 max=500 avg=405
height: min=50 max=500 avg=362


In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 4 # adjust based on vram usage

In [ ]:
from keras.utils import image_dataset_from_directory

from torch.utils.data import Dataset, DataLoader
from PIL import Image

class CatDogDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
    
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]['filepath']).convert('RGB')
        label = self.df.iloc[idx]['label']
        if self.transform:
            img = self.transform(img)
        return img, label

In [ ]:
import torchvision.transforms as T
train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),  
    T.RandomGrayscale(p=0.05),                                              
    T.ToTensor(),
    T.Normalize([0.485, .456, .406], [.229, .224, .225]),
    T.RandomErasing(p=0.2),                                                 
])

split = int(.8 * len(train_df))
train_ds = CatDogDataset(train_df[:split], transform=train_transform)
val_ds = CatDogDataset(train_df[split:], transform=T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, .456, .406], [.229, .224, .225])
]))

train_loader = DataLoader(train_ds, batch_size= BATCH_SIZE, shuffle = True, num_workers=8, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=8, pin_memory=True)

In [ ]:
""" import torch.nn as nn
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.SiLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels)
        )
        self.silu = nn.SiLU()
    def forward(self, x):
        return self.silu(x+self.block(x)) """

' import torch.nn as nn\nclass ResBlock(nn.Module):\n    def __init__(self, channels):\n        super().__init__()\n        self.block = nn.Sequential(\n            nn.Conv2d(channels, channels, 3, padding=1),\n            nn.BatchNorm2d(channels),\n            nn.SiLU(),\n            nn.Conv2d(channels, channels, 3, padding=1),\n            nn.BatchNorm2d(channels)\n        )\n        self.silu = nn.SiLU()\n    def forward(self, x):\n        return self.silu(x+self.block(x)) '

In [ ]:
import torch.nn as nn
class SEBlock(nn.Module):
    def __init__(self, channels, r=16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // r),
            nn.SiLU(),
            nn.Linear(channels // r, channels),
            nn.Sigmoid(),
        )
    
    def forward(self, x):
        return x * self.se(x).view(-1, x.shape[1], 1, 1)

In [ ]:
class MBConv(nn.Module):
    def __init__(self, channels, expand=4):
        super().__init__()
        mid = channels * expand
        self.block = nn.Sequential(
            #expand
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.BatchNorm2d(mid),
            nn.SiLU(),

            nn.Conv2d(mid, mid, 3, padding=1, groups=mid, bias=False),
            nn.BatchNorm2d(mid),
            nn.SiLU(),

            SEBlock(mid),

            nn.Conv2d(mid, channels, 1, bias=False),
            nn.BatchNorm2d(channels),
        )

    def forward(self, x):
        return x + self.block(x)

In [ ]:
class HaydenNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1), # 3 channels, 32 filters,
            nn.BatchNorm2d(64),
            nn.SiLU(),
            MBConv(64),
            MBConv(64),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1), # 32 inputs to match previous layer output of 32
            nn.BatchNorm2d(128),
            nn.SiLU(),
            MBConv(128),
            MBConv(128),
            MBConv(128),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.SiLU(),
            MBConv(256),
            MBConv(256),
            MBConv(256),
            SEBlock(256),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512), nn.SiLU(),
            MBConv(512), MBConv(512),
            SEBlock(512),
            nn.MaxPool2d(2), 
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512,256),
            nn.BatchNorm1d(256),
            nn.SiLU(),
            nn.Dropout(.3),
            nn.Linear(256,128),
            nn.BatchNorm1d(128),
            nn.SiLU(),
            nn.Dropout(.3),
            nn.Linear(128,1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
import torchvision.models as models
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device', device)

""" model = models.efficientnet_b0(weights='IMAGENET1K_V1')

for param in model.parameters():
    param.requires_grad = False

model.classifier = nn.Sequential(
    nn.Dropout(.3),
    nn.Linear(model.classifier[1].in_features, 1),
    nn.Sigmoid()
)
model = model.to(device)
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=.001)
 """

model = HaydenNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=.001)




Device cuda


In [ ]:
#from torchinfo import summary
#summary(model, input_size=(1, 3, 224, 224))

In [ ]:
from tqdm import tqdm

criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)

best_acc = 0

for epoch in range(100):
    #print("starting epoch ", epoch)
    model.train()
    train_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/100")
    for imgs, labels in loop:
        imgs, labels = imgs.to(device), labels.float().to(device)
        optimizer.zero_grad()
        preds = model(imgs).squeeze()
        loss = criterion(preds, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

        loop.set_postfix(loss=f"{loss.item():.4f}")

    model.eval()
    correct, total = 0,0
    val_loss = 0
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc="  Validating", leave=False):
            imgs, labels = imgs.to(device), labels.float().to(device)
            preds = model(imgs).squeeze()
            val_loss += criterion(preds, labels).item()
            correct += ((preds > 0) == labels).sum().item()
            total += len(labels)
    acc = correct / total
    scheduler.step(val_loss)
    print(f'Epoch {epoch+1} | Val Acc: {acc:.4f} | LR: {optimizer.param_groups[0]["lr"]:.2e}')

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f'  -> saved new best ({best_acc:.4f})')

Epoch 1/20:  43%|████▎     | 2162/5000 [06:17<08:15,  5.73it/s, loss=0.6213]


KeyboardInterrupt: 

In [ ]:
#hot start


In [ ]:
test_paths = sorted(Path('./data/test').glob('*jpg'), key=lambda p: int(p.stem))

test_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, .456, .406], [.229, .224, .225])
])

ids, preds = [], []
model.eval()
with torch.no_grad():
    for path in test_paths:
        img = Image.open(path).convert("RGB")
        img = test_transform(img).unsqueeze(0).to(device)
        prob = torch.sigmoid(model(img)).item()
        ids.append(int(path.stem))
        preds.append(prob)

submission = pd.DataFrame({'id': ids, 'label':preds})
submission.to_csv('submission.csv', index=False)

In [ ]:
print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  33497 KiB |   5987 MiB | 395192 GiB | 395192 GiB |
|       from large pool |  26048 KiB |   5976 MiB | 394988 GiB | 394988 GiB |
|       from small pool |   7449 KiB |     13 MiB |    204 GiB |    204 GiB |
|---------------------------------------------------------------------------|
| Active memory         |  33497 KiB |   5987 MiB | 395192 GiB | 395192 GiB |
|       from large pool |  26048 KiB |   5976 MiB | 394988 GiB |